In [ ]:
%pwd

'/Users/earthpatel/Desktop/Portfolio/investment_platform/notebooks'

In [ ]:
import yfinance as yf
import pandas as pd
import os

# -------------------------------
# 1. Loading the CSV
# -------------------------------
file_path = "../data/raw/sp500_tickers.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"Could not find {file_path}. Check your folder structure!")

tickers_df = pd.read_csv(file_path)

# Normalizing the ticker column
column_name = 'Symbol' if 'Symbol' in tickers_df.columns else 'Ticker'
tickers = tickers_df[column_name].str.replace('.', '-', regex=False).tolist()

# -------------------------------
# 2. Building lazy-loading stock database
# -------------------------------
stock_data = {}

for ticker in tickers:
    t_obj = yf.Ticker(ticker)
    
    # Using lambda with default argument to preserve the current Ticker object
    stock_data[ticker] = {
        'history': lambda obj=t_obj: obj.history(period="5y"),
        'info': lambda obj=t_obj: obj.info,
        'financials': lambda obj=t_obj: obj.financials
    }

print(f"Database initialized with {len(stock_data)} tickers.")

# -------------------------------
# 3. Example usage
# -------------------------------
example_ticker = 'AAPL'

# Get 5-year history
history_df = stock_data[example_ticker]['history']()
print(f"\n{example_ticker} 5-year history:")
print(history_df.tail())

# Get company info
info = stock_data[example_ticker]['info']()
print(f"\n{example_ticker} info:")
print({k: info[k] for k in list(info)[:5]})  # Print first 5 fields

# Get financials
financials_df = stock_data[example_ticker]['financials']()
print(f"\n{example_ticker} financials:")
print(financials_df.head())

Database initialized with 503 tickers.

AAPL 5-year history:
                                 Open        High         Low       Close  \
Date                                                                        
2026-03-20 00:00:00-04:00  247.979996  249.199997  246.000000  247.990005   
2026-03-23 00:00:00-04:00  253.970001  254.600006  250.279999  251.490005   
2026-03-24 00:00:00-04:00  250.350006  254.830002  249.550003  251.639999   
2026-03-25 00:00:00-04:00  254.100006  255.000000  251.600006  252.619995   
2026-03-26 00:00:00-04:00  251.994995  257.000000  250.774994  254.365005   

                             Volume  Dividends  Stock Splits  
Date                                                          
2026-03-20 00:00:00-04:00  88331100        0.0           0.0  
2026-03-23 00:00:00-04:00  40546100        0.0           0.0  
2026-03-24 00:00:00-04:00  45152300        0.0           0.0  
2026-03-25 00:00:00-04:00  28436700        0.0           0.0  
2026-03-26 00:00:00-0

In [ ]:
ticker_keys = list(stock_data.keys())
print(ticker_keys)

['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APP', 'APTV', 'ACGL', 'ADM', 'ARES', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVB', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK-B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BK', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF-B', 'BLDR', 'BG', 'BXP', 'CHRW', 'CDNS', 'CPT', 'CPB', 'COF', 'CAH', 'CCL', 'CARR', 'CVNA', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CIEN', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'COHR', 'COIN', 'CL', 'CMCSA', 'FIX', 'CAG', 'COP', 'ED', 'STZ', 'CEG', 'COO', 'CPRT', 'GLW', 'CPAY', 'CTVA', 'CSGP', 'COST', 'CTRA', 'CRH', 'CRWD', 'CCI', 'CSX'

In [ ]:
# Making a flattened history database

tickers = list(stock_data.keys())
all_history = pd.DataFrame()

for ticker in tickers[:10]:  # just first 10 tickers for now
    try:
        df = stock_data[ticker]['history']()
        if df.empty:
            print(f"No data for {ticker}")
            continue
        df['Ticker'] = ticker
        all_history = pd.concat([all_history, df])
    except Exception as e:
        print(f"Error fetching {ticker}: {e}")

all_history.reset_index(inplace=True)  # make Date a column
print(all_history.shape)  # see how many rows × columns
print(all_history.head()) # first 5 rows

(12560, 9)
                       Date        Open        High         Low       Close  \
0 2021-03-26 00:00:00-04:00  134.725135  136.079306  134.166727  136.030441   
1 2021-03-29 00:00:00-04:00  135.150938  137.314798  135.074150  136.630737   
2 2021-03-30 00:00:00-04:00  136.100280  137.252024  135.332464  135.862961   
3 2021-03-31 00:00:00-04:00  135.374320  135.716350  134.055052  134.494812   
4 2021-04-01 00:00:00-04:00  134.892676  135.360359  133.210448  134.508774   

    Volume  Dividends  Stock Splits Ticker  
0  3202410        0.0           0.0    MMM  
1  3111753        0.0           0.0    MMM  
2  2244533        0.0           0.0    MMM  
3  2902572        0.0           0.0    MMM  
4  2274912        0.0           0.0    MMM  


In [ ]:
# Making a flattened info database

all_info = pd.DataFrame()

for ticker in tickers[:10]:  # test with 10 tickers first
    try:
        info_dict = stock_data[ticker]['info']()  # call the lambda
        if not info_dict:  # skip empty
            print(f"No info for {ticker}")
            continue
        info_dict['Ticker'] = ticker
        all_info = pd.concat([all_info, pd.DataFrame([info_dict])], ignore_index=True)
    except Exception as e:
        print(f"Error fetching info for {ticker}: {e}")

print(all_info.shape)
print(all_info.head())

(10, 182)
                address1           city state         zip        country  \
0              3M Center     Saint Paul    MN  55144-1000  United States   
1  11270 West Park Place      Milwaukee    WI       53224  United States   
2   100 Abbott Park Road    Lake County    IL  60064-6400  United States   
3  1 North Waukegan Road  North Chicago    IL  60064-6400  United States   
4   1 Grand Canal Square         Dublin   NaN    D02 P820        Ireland   

            phone                    website                         industry  \
0    651 733 1110         https://www.3m.com                    Conglomerates   
1    414 359 4000    https://www.aosmith.com   Specialty Industrial Machinery   
2    224 667 6100     https://www.abbott.com                  Medical Devices   
3    847 932 7900     https://www.abbvie.com     Drug Manufacturers - General   
4  353 1 646 2000  https://www.accenture.com  Information Technology Services   

                       industryKey            

In [ ]:
all_financials = pd.DataFrame()
tickers = list(stock_data.keys())

for ticker in tickers[:10]:  # test small batch
    try:
        fin_df = stock_data[ticker]['financials']()
        if fin_df.empty:
            print(f"No financials for {ticker}")
            continue

        # Transpose so columns become rows (historical periods)
        fin_df_T = fin_df.T

        # Add Ticker column
        fin_df_T['Ticker'] = ticker

        # Reset index to have 'Year' column
        fin_df_T = fin_df_T.reset_index().rename(columns={'index':'Year'})

        # Concatenate to master DataFrame
        all_financials = pd.concat([all_financials, fin_df_T], ignore_index=True)

    except Exception as e:
        print(f"Error fetching financials for {ticker}: {e}")

print(f"Final financials table shape: {all_financials.shape}")
print(all_financials.head())


Final financials table shape: (10, 69)
   Tax Effect Of Unusual Items  Tax Rate For Calcs  Normalized EBITDA  \
0                 5.713743e+07            0.238073       6.227000e+09   
1                 0.000000e+00            0.236000       8.137000e+08   
2                 1.145000e+07            0.229000       1.202500e+10   
3                -1.818241e+09            0.358345       2.270300e+10   
4                 0.000000e+00            0.237381       1.186733e+10   

   Total Unusual Items  Total Unusual Items Excluding Goodwill  \
0         2.400000e+08                            2.400000e+08   
1         0.000000e+00                            0.000000e+00   
2         5.000000e+07                            5.000000e+07   
3        -5.074000e+09                           -5.074000e+09   
4                  NaN                                     NaN   

   Net Income From Continuing Operation Net Minority Interest  \
0                                       3.250000e+09        

In [ ]:
# Storing each processed dataframes 

all_history.to_csv("../data/processed/all_history.csv", index=False)

if 'Ticker' in all_history.columns:
    cols = ['Ticker'] + [c for c in all_history.columns if c != 'Ticker']
    all_history = all_history[cols]
all_history.to_csv("../data/processed/all_history.csv", index=False)

if 'Ticker' not in all_info.columns:
    all_info['Ticker'] = list(stock_data.keys())[:len(all_info)]
cols = ['Ticker'] + [c for c in all_info.columns if c != 'Ticker']
all_info = all_info[cols]
all_info.to_csv("../data/processed/all_info.csv", index=False)

if 'Ticker' not in all_financials.columns:
    all_financials['Ticker'] = list(stock_data.keys())[:len(all_financials)]
cols = ['Ticker'] + [c for c in all_financials.columns if c != 'Ticker']
all_financials = all_financials[cols]
all_financials.to_csv("../data/processed/all_financials.csv", index=False)

print("All processed CSVs saved with Ticker as the first column!")

All processed CSVs saved with Ticker as the first column!


In [ ]:
# Merge info and financials into history
master_df = all_history.merge(all_info, on='Ticker', how='left') \
                       .merge(all_financials, on='Ticker', how='left')

master_df.to_csv("../data/processed/master_dataset.csv", index=False)